# Milestone - Unit 9: feature towers and item cold-start -- the Unit-8 two-tower rebuilt so the item side is made of *content* (a learned id embedding + learned genre and author embeddings + a `Linear` projection of the Unit-7 GloVe keyword vector), combined to the same 32-dim space and scored by the same `reader . item` dot product. Because those feature parameters are shared across every book and trained from the warm items, a **cold book** -- zero train-positive interactions -- still gets a sensible vector, so the feature tower is the first **learned-taste / collaborative** path in the book to surface cold items (coverage 0.138 over the 818 zero-train books, where the ID-only two-tower and item-item CF are exactly 0) while holding its warm hit@10 near the book's best (0.298 vs the two-tower's 0.340). A warm/cold **compromise**, not dominance.

## 1. Load the catalog, interactions, cold readers, and GloVe

The same generated substrate Units 1-8 ran on -- the catalog, the interaction log (train / val / test splits with implicit 0/1 labels) and the committed cold-reader partition every scoreboard excludes -- plus the Unit-7 GloVe subset, which the feature tower now needs because its item tower reads each book's keyword vector. Two quantities are defined up front and are deliberately **validation-free**: the cold denominator is the 818 **zero-train** books (no train positive, hence no learned id row -- only features can place them), and the coverage cohort is every fitted non-cold reader scored from TRAIN history only, so the cold metric never depends on who happens to have a val positive. `bookrec.generated_dir()` reports a clear error if `recsys/data/generated/` has not been built.

In [1]:
import json
from collections import defaultdict

import bookrec
import pandas as pd
from bookrec import (
    FeatureTowerRetrievalPath,
    ItemItemRetrievalPath,
    LexicalRetrievalPath,
    MatrixFactorizationPath,
    PathRegistry,
    PopularityRetrievalPath,
    SemanticEmbeddingRetrievalPath,
    TwoTowerRetrievalPath,
    load_glove_subset,
    run_blended_scoreboard,
    run_validation_scoreboard,
)

K = 10
POOL = 30
SEED = 0

data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")
glove = load_glove_subset()

cold_partition = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = cold_partition["cold_readers"]
cold_reader_set = set(cold_readers)

interactions_path = data_dir / "interactions.csv.gz"
rows = pd.read_csv(interactions_path).to_dict("records")

# Train-seen items per reader, plus the ZERO-TRAIN items -- the validation-free cold denominator.
# A cold book has no train positive, so no learned id embedding is ever trained for it; only its
# features can place it. val positives are read ONLY to report the incidental-cold-val cohort later.
seen_by_reader = defaultdict(set)
val_by_reader = defaultdict(set)
train_positive_items = set()
for row in rows:
    if int(row["label"]) != 1:
        continue
    if row["split"] == "train":
        seen_by_reader[row["reader_id"]].add(row["item_id"])
        train_positive_items.add(row["item_id"])
    elif row["split"] == "val":
        val_by_reader[row["reader_id"]].add(row["item_id"])
zero_train_items = set(catalog_ids) - train_positive_items

# The coverage cohort: ALL fitted non-cold readers (>= 1 train positive, minus the held-out cold
# readers), scored from TRAIN history only, so coverage never depends on who has a val positive.
coverage_cohort = [r for r in sorted(seen_by_reader) if r not in cold_reader_set]

{
    "books": len(catalog_ids),
    "interactions": len(rows),
    "zero_train_items": len(zero_train_items),
    "coverage_cohort": len(coverage_cohort),
    "cold_readers_excluded": len(cold_readers),
}

{'books': 2000,
 'interactions': 27004,
 'zero_train_items': 818,
 'coverage_cohort': 540,
 'cold_readers_excluded': 60}

## 2. Fit the six earlier paths and train the feature tower (three seeded fits)

Seven signals, each a `RetrievalPath` with the identical `fit` / `retrieve` contract: popularity, the Unit-3 lexical path, the Unit-4 item-item CF, the Unit-7 semantic (GloVe) path, the Unit-5 matrix factorization, the Unit-8 ID-only two-tower, and the new Unit-9 **feature tower**. Three do real numeric training -- MF (~15 s), the ID-only two-tower (~15 s) and the feature tower (~25 s at 40 epochs) -- three seeded fits, every RNG fixed, so the whole demo reproduces and stays in CI budget. The feature item tower combines four signals to the shared 32-dim space: a learned per-id embedding (the Unit-8 collaborative row), a mean-pooled genre embedding, an author embedding, and a `Linear` projection of the book's GloVe keyword vector. The reader tower stays a bare id embedding and the score is still `reader . item`. The decisive knob is `negative_pool="warm"`: negatives are drawn **only** from the train-positive item universe, so a cold book is never sampled as a negatives-only id row and its features are free to place it. torch is imported lazily inside `fit`, seeded, single-threaded and deterministic, then its process globals are restored. We confirm the BPR loss fell over training (weight decay keeps it from collapsing to 0).

In [2]:
popularity = PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
lexical = LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)
cf = ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)
semantic = SemanticEmbeddingRetrievalPath(keywords, glove).fit(rows, catalog=catalog_ids)
mf = MatrixFactorizationPath(seed=SEED).fit(rows, catalog=catalog_ids)  # ~15 s
two_tower = TwoTowerRetrievalPath(seed=SEED).fit(rows, catalog=catalog_ids)  # ~15 s, ID-only item tower

# The feature tower: item side = id + genre + author + a Linear projection of the GloVe keyword
# vector; warm-only negatives so a cold book's features can place it (THE design choice).
feature_tower = FeatureTowerRetrievalPath(
    catalog, keywords, glove, n_epochs=40, negative_pool="warm", seed=SEED
).fit(rows, catalog=catalog_ids)  # ~25 s

registry = PathRegistry()
for path in (popularity, lexical, cf, semantic, mf, two_tower, feature_tower):
    registry.register(path)

# BPR falls over training; weight decay keeps it from memorizing (it does NOT collapse to 0).
first_loss, last_loss = feature_tower.loss_history[0], feature_tower.loss_history[-1]
assert last_loss < first_loss

{
    "registered_paths": [path.name for path in registry],
    "reader_tower": list(feature_tower.reader_embeddings.shape),
    "item_tower": list(feature_tower.item_embeddings.shape),
    "bpr_loss_first_epoch": round(first_loss, 4),
    "bpr_loss_last_epoch": round(last_loss, 4),
}

{'registered_paths': ['popularity',
  'lexical',
  'item-item',
  'semantic',
  'mf',
  'two-tower',
  'feature-tower'],
 'reader_tower': [540, 32],
 'item_tower': [2000, 32],
 'bpr_loss_first_epoch': 0.2188,
 'bpr_loss_last_epoch': 0.0688}

## 3. The `val` scoreboard -- warm hit stays near the book's best

We score all seven paths on the frozen `val` split with `run_validation_scoreboard` (the same 500 eligible readers, the 60 cold readers excluded -- never `test`, which Checkpoint B owns). Read the feature tower honestly: at hit@10 ~0.298 it does **not** beat the ID-only two-tower (0.340) -- rebuilding the item tower from features costs a little warm accuracy -- but it stays within 0.06 of the book's best and above every non-neural path, MF included. The payoff is the next section: cold reach the two-tower simply does not have.

In [3]:
warm_rows = []
for path in registry:
    board = run_validation_scoreboard(
        path, interactions_path, catalog_ids=catalog_ids, k=K, cold_readers=cold_readers
    )
    warm_rows.append({
        "path": path.name,
        "hit@10": round(board.hit_rate_at_k, 4),
        "precision@10": round(board.precision_at_k, 4),
        "ndcg@10": round(board.ndcg_at_k, 4),
    })

warm = {entry["path"]: entry["hit@10"] for entry in warm_rows}

# Ground-truth val hit@10 (seed 0, k=10, 500 eligible readers) bound to the shipped paths.
assert abs(warm["popularity"] - 0.108) < 0.001
assert abs(warm["lexical"] - 0.158) < 0.001
assert abs(warm["semantic"] - 0.102) < 0.001
assert abs(warm["item-item"] - 0.252) < 0.001
assert abs(warm["mf"] - 0.276) < 0.001
assert abs(warm["two-tower"] - 0.340) < 0.001
assert abs(warm["feature-tower"] - 0.298) < 0.001
# The feature tower is a warm/cold COMPROMISE: it does NOT beat the ID-only two-tower on warm hit
# (0.298 < 0.340), yet it stays within 0.06 of the book's best and above every non-neural path,
# MF included -- so the cold reach of the next section is bought with a small warm cost, not a tank.
assert warm["two-tower"] == max(entry["hit@10"] for entry in warm_rows)
assert warm["feature-tower"] >= 0.28
assert warm["feature-tower"] >= warm["two-tower"] - 0.06
assert warm["feature-tower"] > warm["mf"]

pd.DataFrame(warm_rows)

,path,hit@10,precision@10,ndcg@10
0,popularity,0.108,0.0112,0.0332
1,lexical,0.158,0.0176,0.0433
2,item-item,0.252,0.0282,0.0816
3,semantic,0.102,0.0110,0.0325
4,mf,0.276,0.0326,0.1061
5,two-tower,0.340,0.0438,0.1305
6,feature-tower,0.298,0.0358,0.1083


The two-tower is still the single best warm path. The feature tower gives up ~0.04 of warm hit for something the two-tower cannot do at all -- place a book with no interaction history. That trade is the whole point of the unit, and the next section measures what it buys.

## 4. Cold-item coverage -- the metric the ID-only towers score zero on

Cold-item **coverage** = the fraction of the 818 zero-train books that appear in the top-10 of ANY fitted non-cold reader, scored from that reader's TRAIN history only. It needs no cold val positives (the generator leaves cold items without them -- their relevance is sealed for Checkpoint B), so it is validation-safe and deterministic. The ID-only two-tower and item-item CF give a zero-train book a negative-shaped / unseen embedding, so its coverage is exactly **0**. The feature tower reaches **0.138** (113/818) from features alone. And honestly: the Unit-7 **semantic** content path already reaches cold freely (0.328) -- it simply ignores the interaction log and pays for it on warm hit (0.102). So the feature tower is not the *first* path to serve cold; it is the first **learned-taste / collaborative** path to serve cold while holding near the book's best warm hit.

In [4]:
def fitted_cohort(path):
    """ALL fitted non-cold readers. The two towers expose their fitted reader ids in the artifact;
    CF / semantic have no reader embedding, so they fall back to the full train cohort."""
    try:
        fitted = [int(r) for r in path.artifact()["reader_ids"]]
    except KeyError:
        fitted = sorted(seen_by_reader)
    return [r for r in fitted if r not in cold_reader_set]


def cold_coverage(path):
    """Unique zero-train ids in the top-10 of ANY fitted non-cold reader, scored TRAIN-only: a count
    over the 818 zero-train books / 818, with no val dependency."""
    surfaced = set()
    for reader_id in fitted_cohort(path):
        for candidate in path.retrieve(reader_id, {"seen": seen_by_reader[reader_id]}, K):
            if candidate.item_id in zero_train_items:
                surfaced.add(candidate.item_id)
    return len(surfaced) / len(zero_train_items), len(surfaced)


cold_rows = []
for name, path in (
    ("feature-tower", feature_tower),
    ("two-tower", two_tower),
    ("item-item", cf),
    ("semantic", semantic),
):
    cov, surfaced = cold_coverage(path)
    cold_rows.append({"path": name, "cold_coverage": round(cov, 4), "zero_train_surfaced": surfaced})

cold_board = {entry["path"]: entry for entry in cold_rows}

assert len(zero_train_items) == 818  # the validation-free denominator
# ID-only two-tower and item-item CF bury every cold book (negative-shaped / unseen) -> exactly 0.
assert cold_board["two-tower"]["zero_train_surfaced"] == 0
assert cold_board["item-item"]["zero_train_surfaced"] == 0
# The feature tower surfaces cold books FROM THEIR FEATURES: > 0 and strictly above the ID-only base.
assert abs(cold_board["feature-tower"]["cold_coverage"] - 0.1381) < 0.001
assert cold_board["feature-tower"]["zero_train_surfaced"] == 113
assert cold_board["feature-tower"]["cold_coverage"] > cold_board["two-tower"]["cold_coverage"]
# HONEST: a content path (Unit-7 semantic) already reaches cold freely -- the feature tower is the
# first LEARNED-taste / collaborative path to do so, not the first path overall.
assert cold_board["semantic"]["zero_train_surfaced"] == 268
assert cold_board["semantic"]["cold_coverage"] > cold_board["feature-tower"]["cold_coverage"]

pd.DataFrame(cold_rows)

,path,cold_coverage,zero_train_surfaced
0,feature-tower,0.1381,113
1,two-tower,0.0000,0
2,item-item,0.0000,0
3,semantic,0.3276,268


Coverage says the cold books are *reachable*, not that they are *right*. We can only peek at cold relevance through the 97 **incidental** readers who happen to have a zero-train val positive (the 150 designated cold items have none until Checkpoint B). It is small-n and **report-only** -- never a gate -- but directionally the feature tower lands a cold book for a reader the ID-only two-tower never can.

In [5]:
def cold_hit_report(path):
    """REPORT-ONLY cold hit@10 on readers with a zero-train val positive (small-n; not a gate)."""
    hits = []
    for reader_id in sorted(val_by_reader):
        if reader_id in cold_reader_set:
            continue
        seen = seen_by_reader[reader_id]
        cold_relevant = (val_by_reader[reader_id] - seen) & zero_train_items
        if not cold_relevant:
            continue
        recommended = {c.item_id for c in path.retrieve(reader_id, {"seen": seen}, K)}
        hits.append(1.0 if (recommended & cold_relevant) else 0.0)
    return (sum(hits) / len(hits) if hits else 0.0), len(hits)


feature_cold_hit, n_incidental = cold_hit_report(feature_tower)
id_only_cold_hit, _ = cold_hit_report(two_tower)

assert n_incidental == 97  # the incidental-cold-val cohort is fixed by the data
assert feature_cold_hit >= id_only_cold_hit  # not worse than the ID-only base (small-n, report-only)

{
    "incidental_cold_val_readers": n_incidental,
    "feature_tower_cold_hit@10": round(feature_cold_hit, 4),
    "id_only_cold_hit@10": round(id_only_cold_hit, 4),
}

{'incidental_cold_val_readers': 97,
 'feature_tower_cold_hit@10': 0.0103,
 'id_only_cold_hit@10': 0.0}

## 5. One cold book, surfaced by its features

Make it concrete. Reader 129 -- a deterministic pick from the 97 incidental-cold-val readers -- reads book 139 next, a children's title with **zero** train-positive interactions (it appears only as sampled negatives). The ID-only two-tower and item-item CF cannot place book 139 at all (its id row is negative-shaped -- only ever a sampled negative -- and it has no co-occurrence); the feature tower ranks it inside reader 129's top-10 purely from its genre / author / GloVe features. Other zero-train books ride along in the same list.

In [6]:
reader_id = 129
cold_book_id = 139
seen = seen_by_reader[reader_id]

assert cold_book_id in zero_train_items                        # no train history at all
assert cold_book_id in (val_by_reader[reader_id] - seen)       # the reader actually reads it next

recommended = feature_tower.retrieve(reader_id, {"seen": seen}, K)
rank_of_cold_book = None
for position, candidate in enumerate(recommended, start=1):
    if candidate.item_id == cold_book_id:
        rank_of_cold_book = position
assert rank_of_cold_book is not None  # the feature tower surfaces the cold book inside the top 10

# Baseline contrast (executable): the SAME reader's ID-only two-tower and item-item CF top-10 surface
# NO zero-train book at all -- they know the cold book only as something no one has interacted with.
for base_name, base in [("ID-only two-tower", two_tower), ("item-item CF", cf)]:
    base_recs = base.retrieve(reader_id, {"seen": seen}, K)
    assert not any(c.item_id in zero_train_items for c in base_recs), (
        f"{base_name} surfaces no cold book for reader {reader_id}"
    )

book = catalog[cold_book_id]
cold_books_in_top10 = [c.item_id for c in recommended if c.item_id in zero_train_items]

{
    "reader": reader_id,
    "cold_book": cold_book_id,
    "title": book.title,
    "genres": book.fields.get("genres"),
    "author_id": book.fields.get("author_id"),
    "rank_in_top10": rank_of_cold_book,
    "zero_train_books_in_top10": cold_books_in_top10,
}

{'reader': 129,
 'cold_book': 139,
 'title': 'Book 00139',
 'genres': 'children',
 'author_id': '173',
 'rank_in_top10': 2,
 'zero_train_books_in_top10': [139, 1160, 900]}

## 6. Add the feature tower to the Unit-6 blend -- hit up, coverage flat

Finally, drop the feature tower (weight 0.5) into the Unit-6 pinned blend that Checkpoint B froze -- weights `{popularity 0.25, lexical 0.5, item-item 1.0, mf 1.0}` at per-path pool 30. Like the Unit-8 two-tower it is a strong, confident signal, so it lifts the blend's hit@10 (0.306 -> 0.326) -- more than the two-tower did -- while barely moving catalog coverage (0.333 -> 0.327). The blend takes the warm accuracy; the feature tower's cold reach is a property of the standalone path, not of the pooled top-10.

In [7]:
PINNED_WEIGHTS = {"popularity": 0.25, "lexical": 0.5, "item-item": 1.0, "mf": 1.0}
blend_paths = {"popularity": popularity, "lexical": lexical, "item-item": cf, "mf": mf}

board_without = run_blended_scoreboard(
    blend_paths, interactions_path, catalog_ids=catalog_ids, k=K, pool=POOL,
    weights=PINNED_WEIGHTS, cold_readers=cold_readers,
)
board_with = run_blended_scoreboard(
    {**blend_paths, "feature-tower": feature_tower}, interactions_path, catalog_ids=catalog_ids,
    k=K, pool=POOL, weights={**PINNED_WEIGHTS, "feature-tower": 0.5}, cold_readers=cold_readers,
)

# Measured (Unit-6 pinned blend, pool 30): hit 0.306 -> 0.326, coverage 0.333 -> 0.327.
assert abs(board_without.hit_rate_at_k - 0.306) < 0.003
assert abs(board_without.catalog_coverage - 0.333) < 0.003
assert abs(board_with.hit_rate_at_k - 0.326) < 0.003
assert abs(board_with.catalog_coverage - 0.327) < 0.003
assert board_with.hit_rate_at_k > board_without.hit_rate_at_k  # a strong signal lifts blend hit

print(f"blend WITHOUT feature tower: hit@10={board_without.hit_rate_at_k:.4f} "
      f"coverage={board_without.catalog_coverage:.4f}")
print(f"blend WITH    feature tower: hit@10={board_with.hit_rate_at_k:.4f} "
      f"coverage={board_with.catalog_coverage:.4f}")

blend WITHOUT feature tower: hit@10=0.3060 coverage=0.3330
blend WITH    feature tower: hit@10=0.3260 coverage=0.3270


**The honest headline.** The feature tower rebuilds the two-tower's item side from content -- id + genre + author + a GloVe projection -- so a book with zero interaction history still gets a vector. It is the first **learned-taste / collaborative** path in the book to surface cold items (coverage 0.138 vs the ID-only two-tower's and CF's 0), bought with a small, honest warm cost (hit@10 0.298 vs 0.340) -- a warm/cold **compromise**, not dominance, and its cold reach (0.138) is still below the Unit-7 content path's (0.328). This closes the cold-start gap Unit 6 only flagged.

**The bridge.** Retrieval still scores every one of the catalog's books by brute force. Unit 10 makes that fast with approximate nearest neighbours, Unit 11 adds a learned reranker over the retrieved pool, and Unit 13 returns to cold-start in the ethics / beyond-accuracy thread.